# Notebook 2 — Entity Normalisation: Food, Species and Location
### RapidFuzz fuzzy matching · domain-informed manual mapping · first-listed venue extraction

**Goal:** Preserve the 18,839 retained source records and missing outcome values while applying the project's historical Food, Species and Location grouping rules.

**Input:** `data/cleaned_data.csv` — 18,839 records × 9 columns, with unreported Hospitalizations/Fatalities retained as `NaN`.
**Output:** `data/cleaned_data_grouped.csv` — same records + `Food_grouped`, `Species_grouped`, `primary_location` (12 columns).

**Method:** RapidFuzz token_sort_ratio Food threshold 70 / Species threshold 85, followed by manual grouping rules, and a first-listed Location heuristic. Existing mapping/threshold policy is intentionally unchanged for this scientific integrity correction.

**Limitations:** A grouped Food or Species value is an analytical category, not laboratory proof of the outbreak source. First-listed venue is a proxy label, not a verified site of exposure. These surveillance records are not necessarily unique epidemiological events.

## 1. Setup

In [ ]:
import pandas as pd
from rapidfuzz import process, fuzz

df = pd.read_csv("data/cleaned_data.csv")
original_outcomes = df[["Illnesses", "Hospitalizations", "Fatalities"]].copy()
assert len(df) == 18839 and len(df.columns) == 9, "Unexpected NB1 output; verify source data"
assert df[["Hospitalizations", "Fatalities"]].isna().sum().to_dict() == {
    "Hospitalizations": 3560, "Fatalities": 3538
}, "The original unreported outcomes must remain missing"

print(f"Input: {len(df):,} retained records × {df.shape[1]} columns")
print(f"Food unique strings:    {df['Food'].nunique():,}")
print(f"Species unique strings: {df['Species'].nunique():,}")
print(f"Location unique strings: {df['Location'].nunique():,}")
print(f"Hospitalization outcomes unreported: {df['Hospitalizations'].isna().sum():,}")
print(f"Fatality outcomes unreported: {df['Fatalities'].isna().sum():,}")

compound_n = df['Location'].str.contains(';', na=False).sum()
print(f"Multi-venue Location records: {compound_n:,} ({compound_n/len(df)*100:.1f}%)")
print("Sample multi-venue strings:")
print(df[df['Location'].str.contains(';', na=False)]['Location']
      .value_counts().head(5).to_string())

## 2. Fuzzy matching utility

Reused for both Food and Species — single implementation, two applications.

In [ ]:
def fuzzy_group(series: pd.Series, threshold: int) -> dict:
    """
    Build a mapping {original_string: canonical_string} using RapidFuzz
    token_sort_ratio. Strings scoring >= threshold are mapped to the first
    seen variant in that cluster.

    Parameters
    ----------
    series    : pd.Series — column to normalise
    threshold : int — similarity cutoff (0–100). Higher = stricter.
                Food uses 70 (more variant spellings expected).
                Species uses 85 (scientific names are more regular).

    Returns
    -------
    dict mapping each unique string to its canonical form.
    """
    unique_names = series.unique()
    mapping = {}
    for name in unique_names:
        if name in mapping:
            continue
        matches = process.extract(
            name, unique_names,
            scorer=fuzz.token_sort_ratio,
            limit=None
        )
        similar = [m for m, score, _ in matches if score >= threshold]
        for s in similar:
            mapping[s] = name
    return mapping


def apply_manual_map(df: pd.DataFrame, col: str, manual: dict) -> pd.DataFrame:
    """Apply a manual override mapping and print how many values changed."""
    before = df[col].value_counts().to_dict()
    df[col] = df[col].replace(manual)
    changed = sum(1 for k in manual if k in before)
    print(f"  {changed}/{len(manual)} keys matched and replaced")
    return df

## 3. Food grouping

### 3.1 Fuzzy auto-clustering (threshold = 70)

A lower threshold (70) is appropriate for food names because the same food item
appears with highly variable phrasing across reporters and years.

In [ ]:
# Normalise whitespace before fuzzy matching
df["Food"] = df["Food"].astype(str).str.strip()

food_mapping = fuzzy_group(df["Food"], threshold=70)
df["Food_grouped"] = df["Food"].map(food_mapping)

print(f"Food unique strings:  {df['Food'].nunique():,}  → after fuzzy: {df['Food_grouped'].nunique():,}")
print()
df["Food_grouped"].value_counts().head(20)

### 3.2 Manual mapping — domain corrections and analytical decisions

Fuzzy matching clusters by string similarity, not biological or culinary meaning.
A second pass of manual corrections is required for cases where string similarity
alone cannot determine the appropriate category.

**Two types of corrections — and an important distinction:**

**Type A — Unambiguous normalisation** (string variants of the same category):
- "Greek Salad", "Caesar Salad", "House Salad" → `Salad`
- "Chicken, Fried", "Chicken, BBQ", "Chicken, Nuggets" → `Chicken`
- "Ground Beef, Hamburger", "Ground Beef, Cheeseburger" → `Ground Beef`

These are straightforward: the food item is unambiguously the same category
regardless of preparation method.

**Type B — Analytical decisions under uncertainty** (multi-ingredient dishes):
- "Chicken Salad" → `Chicken`
- "Potato Salad" → `Salad`
- "Fruit Punch, Unspecified" → `Fruit`

These require an explicit caveat: **grouping multi-ingredient dishes into a
single primary category is an analytical simplification, not a microbiological
determination.**

In "Chicken Salad", the contamination source could be the chicken, the
mayonnaise, the celery, or cross-contamination during preparation.
Without laboratory confirmation — which FDOSS does not require for reporting —
the true vehicle is unknown. The mapping decision (chicken as primary vehicle)
reflects a categorisation choice for aggregation purposes only.

This is a known limitation of food vehicle surveillance data generally:
the reported food is the *epidemiologically implicated vehicle*, not necessarily
the *microbiologically confirmed source*. Type B mappings introduce systematic
ambiguity that should be considered when interpreting food-vehicle rankings,
particularly for mixed/prepared dishes.

In [ ]:
FOOD_MANUAL = {
    # Chicken variants
    "Chicken, Fried":                           "Chicken",
    "Chicken Salad":                            "Chicken",
    "Chicken Nachos":                           "Chicken",
    "Chicken, Teriyaki; Sushi, Unspecified":    "Chicken",
    "Chicken, Bbq":                             "Chicken",
    "Chicken; Ribs":                            "Chicken",
    "Chicken, Buffalo Wings":                   "Chicken",
    "Chicken, Nuggets/Fingers":                 "Chicken",
    "Chicken, Confit":                          "Chicken",

    # Fish variants
    "Fish, Ahi":                                "Fish",
    "Fish, Mahi Mahi":                          "Fish",
    "Salmon, Raw; Sushi, Unspecified":          "Fish",
    "Fish, Escolar":                            "Fish",
    "Fish, Baracuda":                           "Fish",
    "Fish, Trout":                              "Fish",
    "Fish, Grouper, Unspecified":               "Fish",
    "Fish, Amberjack":                          "Fish",
    "Fish, Amberjack ":                         "Fish",
    "Unspecified Fish; Applesauce":             "Fish",

    # Salad variants
    "Salad, Unspecified":                       "Salad",
    "Pasta-Based Salads Unspecified; Chicken Salad": "Salad",
    "Potato Salad":                             "Salad",
    "Fruit Salad":                              "Salad",
    "House Salad":                              "Salad",
    "Greek Salad":                              "Salad",
    "Caesar Salad":                             "Salad",

    # Sandwich variants
    "Sandwich, Reuben; Pudding, Unspecified":   "Sandwich",
    "Sandwich, Gyro":                           "Sandwich",
    "Sandwich, Submarine":                      "Sandwich",
    "Sandwich, Turkey; Cantaloupe":             "Sandwich",
    "Sandwich, Cuban":                          "Sandwich",

    # Ground beef
    "Ground Beef, Hamburger":                   "Ground Beef",
    "Ground Beef, Cheeseburger":                "Ground Beef",
    "Ground Beef, Meatballs":                   "Ground Beef",

    # Beef / Pork / Turkey
    "Beef, Bbq; Ham, Unspecified":              "Beef",
    "Beef, Other":                              "Beef",
    "Gravy, Unspecified; Roast Beef, Unspecified; Pork, Roasted": "Beef",
    "Steak, Unspecified; Chicken, Cacciatore; Eggplant, Unspecified": "Steak",
    "Steak, Prime Rib":                         "Steak",
    "Pork, Bbq":                                "Pork",
    "Pork, Roasted":                            "Pork",
    "Pork, Other":                              "Pork",
    "Turkey, Smoked; Ham":                      "Turkey",
    "Turkey, Roasted":                          "Turkey",

    # Dairy / produce / other
    "Fruit Punch, Unspecified":                 "Fruit",
    "Banana, Unspecified; Cantaloupe; Pineapple, Unspecified": "Fruit",
    "Leaf Lettuce, Unspecified":                "Lettuce",
    "Whole Milk, Unpasteurized":                "Milk",
    "Whole Milk, Unpasteurized; Goat Milk, Unpasteurized": "Milk",
    "Chocolate Milk, Unspecified":              "Chocolate Milk",
    "Rice; Beans, Unspecified":                 "Rice/Beans",
    "Rice, Unspecified (Source Unknown)":       "Rice/Beans",
    "Pizza, Meat And Vegetable":                "Pizza",
    "Pizza, Meat":                              "Pizza",
    "Coleslaw; Potato, Unspecified":            "Coleslaw",
    "Cake, Unspecified; Ice Cream, Commercial": "Dessert",
    "Dessert Unspecified":                      "Dessert",
    "Doughnuts, Unspecified":                   "Doughnuts",
    "Butter; Carrots, Unspecified":             "Butter",
    "Pie, Unspecified; Mousse":                 "Pie",
}

print(f"Applying {len(FOOD_MANUAL)} manual corrections...")
df = apply_manual_map(df, "Food_grouped", FOOD_MANUAL)
print(f"Food_grouped unique after manual: {df['Food_grouped'].nunique():,}")
print()
df["Food_grouped"].value_counts().head(15)

### 3.3 Food grouping result

In [ ]:
food_before = df["Food"].nunique()
food_after  = df["Food_grouped"].nunique()
print(f"Food: {food_before:,} unique strings → {food_after:,} grouped categories")
print(f"Reduction: {(1 - food_after/food_before)*100:.0f}%")
print()
print("Top 10 food groups by outbreak frequency:")
print(df[df["Food_grouped"] != "Unknown"]["Food_grouped"].value_counts().head(10).to_string())

## 4. Species grouping

### 4.1 Fuzzy auto-clustering (threshold = 85)

A higher threshold (85) is used for species names because scientific nomenclature is more
regular — we want to cluster only genuinely similar strings, not merge different species.

In [ ]:
df["Species"] = df["Species"].astype(str).str.strip()

species_mapping = fuzzy_group(df["Species"], threshold=85)
df["Species_grouped"] = df["Species"].map(species_mapping)

print(f"Species unique strings: {df['Species'].nunique():,}  → after fuzzy: {df['Species_grouped'].nunique():,}")
print()
df["Species_grouped"].value_counts().head(20)

### 4.2 Manual mapping — taxonomic corrections

Fuzzy matching cannot resolve these cases without biological knowledge:
- All Norovirus genogroup variants → `Norovirus` (same epidemiological unit)
- `Salmonella unknown` vs `Salmonella enterica` — kept separate (species matters for severity)
- `E. coli` subtypes (EPEC, STEC) → `Escherichia coli` for EDA-level aggregation
- Co-infections (e.g., `Bacillus cereus; Clostridium perfringens`) → primary pathogen

In [ ]:
SPECIES_MANUAL = {
    # Norovirus — all genogroup variants are the same epidemiological entity
    "Norovirus genogroup I":                        "Norovirus",
    "Norovirus unknown":                            "Norovirus",
    "Norovirus genogroup I; Norovirus genogroup II": "Norovirus",

    # Salmonella — keep enterica separate from unknown for severity analysis
    "Salmonella unknown":                           "Salmonella",
    "Salmonella enterica; Salmonella enterica":     "Salmonella enterica",

    # E. coli subtypes → aggregate for EDA
    "Escherichia coli, Enteropathogenic":           "Escherichia coli",
    "Escherichia coli, Shiga toxin-producing":      "Escherichia coli",

    # Co-infections → primary pathogen
    "Bacillus cereus; Clostridium perfringens":     "Bacillus cereus",
    "Staphylococcus aureus; Bacillus cereus":       "Staphylococcus aureus",

    # Campylobacter normalisation
    "Campylobacter unknown":                        "Campylobacter",
    "Campylobacter":                                "Campylobacter",
}

print(f"Applying {len(SPECIES_MANUAL)} manual corrections...")
df = apply_manual_map(df, "Species_grouped", SPECIES_MANUAL)
print(f"Species_grouped unique after manual: {df['Species_grouped'].nunique():,}")
print()
df["Species_grouped"].value_counts().head(15)


### 4.3 Repeated-entry normalisation

RapidFuzz and manual mapping handle variant spellings, but cannot resolve
entries where the same species name was concatenated multiple times
(e.g., `"Salmonella enterica; Salmonella enterica; ..."` × up to 13 repetitions).
These arise from data entry errors in the original CDC records.

**Rule:** split on `;`, remove duplicate parts while preserving order,
rejoin. Genuine co-infections with two *different* pathogens
(e.g., `"Bacillus cereus; Norovirus"`) are left unchanged.

In [ ]:
def clean_repeated_species(s: str) -> str:
    """
    Collapse repeated semicolon-separated entries to a single canonical form.

    Examples
    --------
    "Salmonella enterica; Salmonella enterica; Salmonella enterica"
        → "Salmonella enterica"

    "Bacillus cereus; Norovirus"          (genuine co-infection)
        → "Bacillus cereus; Norovirus"    (unchanged)
    """
    parts = [p.strip() for p in str(s).split(";")]
    unique_parts = list(dict.fromkeys(parts))   # deduplicate, preserve order
    return unique_parts[0] if len(unique_parts) == 1 else "; ".join(unique_parts)


before = df["Species_grouped"].nunique()
df["Species_grouped"] = df["Species_grouped"].apply(clean_repeated_species)
after  = df["Species_grouped"].nunique()

print(f"Species_grouped unique: {before} → {after}")
print()
print("Top 15 after cleaning:")
print(df[df["Species_grouped"] != "Unknown"]["Species_grouped"]
      .value_counts().head(15).to_string())

### 4.4 Species grouping result

In [ ]:
species_before = df["Species"].nunique()
species_after  = df["Species_grouped"].nunique()
print(f"Species: {species_before:,} unique strings → {species_after:,} grouped categories")
print(f"Reduction: {(1 - species_after/species_before)*100:.0f}%")
print()
print("Top 10 pathogen groups by outbreak frequency:")
print(df[df["Species_grouped"] != "Unknown"]["Species_grouped"]
      .value_counts().head(10).to_string())

## 5. Location normalisation

### 5.1 Multiple reported venue types

Location strings can list more than one setting separated by semicolons. The original `Location` value is retained and the first listed segment is extracted as a **descriptive proxy** called `primary_location`. The data alone do not verify that it was the true primary exposure site, or establish the order's surveillance meaning.

In [ ]:
# ── Audit before normalisation ───────────────────────────────
loc_vc = df['Location'].value_counts()
n_compound = df['Location'].str.contains(';', na=False).sum()

print(f"Location unique raw strings: {df['Location'].nunique()}")
print(f"Multi-venue records:         {n_compound:,} ({n_compound/len(df)*100:.1f}%)")
print()
print("Top 10 most frequent raw Location values:")
print(loc_vc.head(10).to_string())

### 5.2 First-listed venue extraction

**Analytical rule:** `Location.str.split(";").str[0].str.strip()`, preserved from the original project.

**Interpretation boundary:** This is the first <i>listed</i> venue, not an independently confirmed causal exposure setting. Multi-venue records should not be interpreted as proving outbreak spread among those venues.

**Unmerged categories:** Fast Food Restaurant, Restaurant, Restaurant Buffet, Nursing Home/Assisted Living Facility and Hospital remain distinct labels. Different observed reporting categories do not by themselves demonstrate different HACCP performance or causal mechanisms.

In [ ]:
# ── Primary venue extraction ─────────────────────────────────
df["primary_location"] = df["Location"].str.split(";").str[0].str.strip()

before = df["Location"].nunique()
after  = df["primary_location"].nunique()
print(f"Location: {before} raw unique strings → {after} primary venue types")
print(f"Reduction: {(1 - after/before)*100:.0f}%")
print()
print("primary_location distribution:")
print(df["primary_location"].value_counts().to_string())

### 5.3 Validation and scope note

The original `Location` field remains available alongside the derived first-listed `primary_location`. There are still unknown and multi-venue setting records; `primary_location` is not a validated exposure location. Analyses should identify whether unknown settings were excluded from a particular denominator. Do not interpret differences in setting-level proportions as causal or population-adjusted risk.

In [ ]:
# ── Verification ─────────────────────────────────────────────
# Check no new nulls introduced
nulls = df["primary_location"].isna().sum()
print(f"Null values in primary_location: {nulls}  (expected: 0)")

# Confirm 'Unknown' is preserved correctly
unknown_n = (df["primary_location"] == "Unknown").sum()
original_unknown = (df["Location"].str.split(";").str[0].str.strip() == "Unknown").sum()
print(f"Unknown records: {unknown_n:,}  (original: {original_unknown:,})")

# Show before/after for compound strings
sample_compound = df[df["Location"].str.contains(";", na=False)][
    ["Location", "primary_location"]
].drop_duplicates().head(8)
print()
print("Sample: compound Location → primary_location")
print(sample_compound.to_string(index=False))

## 6. Validation — no data loss

In [ ]:
# Verify preserved cohort, categorical outputs and untouched outcome values.
assert len(df) == 18839, f"Record count changed in NB2: {len(df)}"
assert df.shape[1] == 12, f"Expected 12 columns, got {df.shape[1]}"
for col in ["Food_grouped", "Species_grouped", "primary_location"]:
    assert df[col].notna().all(), f"Unexpected missing grouped category: {col}"

pd.testing.assert_frame_equal(
    df[["Illnesses", "Hospitalizations", "Fatalities"]].reset_index(drop=True),
    original_outcomes.reset_index(drop=True),
    check_exact=True,
)
assert df[["Hospitalizations", "Fatalities"]].isna().sum().to_dict() == {
    "Hospitalizations": 3560, "Fatalities": 3538
}
assert df["Food_grouped"].nunique() > 0 and df["Species_grouped"].nunique() > 0
assert df["primary_location"].nunique() < 30

print(f"Validated: {len(df):,} source-aligned records, 12 columns")
print("All original observed values and unreported outcome NaNs preserved.")
print(f"Food: {df['Food'].nunique():,} → {df['Food_grouped'].nunique():,} groups")
print(f"Species: {df['Species'].nunique():,} → {df['Species_grouped'].nunique():,} groups")
print(f"Location: {df['Location'].nunique():,} → {df['primary_location'].nunique():,} first-listed groups")
df[["Location", "primary_location", "Food", "Food_grouped",
    "Species", "Species_grouped"]].head(5)

## 7. Export

In [ ]:
from pathlib import Path
output_path = Path("data") / "cleaned_data_grouped.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)

# Check serialization roundtrip: numeric NaNs stay missing, not zeros.
reloaded = pd.read_csv(output_path)
assert reloaded.shape == (18839, 12), "Unexpected saved grouped output shape"
pd.testing.assert_frame_equal(reloaded, df, check_dtype=False)
print(f"Saved: {output_path}: {len(df):,} records × {len(df.columns)} columns")
print("Existing Food, Species and Location grouping policies retained.")
print("Next: NB3 and NB4 severity analyses must use outcome-observed denominators.")